# In-Context Learning as Gradient Descent
### The "13-parameter" minimal transformer

**Papers:**
- Oswald et al. (NeurIPS 2023) — [arXiv:2212.07677](https://arxiv.org/abs/2212.07677)
  *Transformers Learn In-Context Learning by Gradient Descent*
- Akyürek et al. (ICLR 2023) — [arXiv:2211.15661](https://arxiv.org/abs/2211.15661)
  *What Learning Algorithm is In-Context Learning? Investigations with Linear Models*

---

## The big picture

**In-context learning (ICL):** A language model sees a few $(x_i, y_i)$ examples in
its prompt, then predicts $y$ for a new $x$ — without any weight updates.

**The insight from Oswald et al.:**
A *single-layer linear attention* transformer can implement this exactly — and the
operation it performs is equivalent to **one step of gradient descent** on the training
examples.

This means:
- Transformers don't need explicit optimisation algorithms
- The attention mechanism *is* an optimiser
- Even a **13-parameter** model can do in-context learning


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from mineo.experiments.icl_gd import (
    ICLGradientDescentExperiment,
    LinearICLModel,
    LinearRegressionTaskGenerator,
    one_step_gd_predict,
    ols_predict,
)
from mineo.visualization.plots import plot_icl_comparison, plot_training_loss

print(f"PyTorch {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


## Mathematical derivation

### The task

Each "task" is a **linear regression problem** with an unknown ground-truth weight $w^*$:

$$y_i = x_i \cdot w^* + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

Given $n$ context pairs $\{(x_1, y_1), \ldots, (x_n, y_n)\}$ and a query $x_*$,
we want to predict $\hat{y}_* \approx x_* \cdot w^*$.

### One step of gradient descent

Starting from $w_0 = \mathbf{0}$, one GD step minimising MSE gives:

$$\nabla_{w} \mathcal{L}\big|_{w=0} = -\frac{1}{n}\sum_i x_i y_i$$

$$w_1 = w_0 - \eta \cdot \nabla_{w} \mathcal{L}\big|_{w=0} = \frac{\eta}{n}\sum_i x_i y_i$$

$$\hat{y}_* = x_* \cdot w_1 = \frac{\eta}{n}\sum_i y_i (x_* \cdot x_i)$$

### Linear attention

Represent each $(x_i, y_i)$ pair as a token $z_i = [x_i;\ y_i] \in \mathbb{R}^2$ (for 1D inputs).
Query token: $z_* = [x_*;\ 0]$ (unknown label, initialised to 0).

Linear (softmax-free) attention computes:

$$\text{Attn}(Z) = \frac{1}{n} Z \cdot (W_K W_Q^\top) \cdot Z^\top \cdot W_V \cdot Z$$

With $W_K = W_Q = I$ and $W_V = -e_1 e_2^\top$ (reads $y$, writes to $x$-slot):

$$\text{output at query} \propto \frac{1}{n}\sum_i y_i (x_* \cdot x_i)$$

**This is exactly the one-step GD prediction.** ✓


## The 13-parameter construction

For 1-dimensional inputs ($x, y \in \mathbb{R}$), the token space is 2D ($z = [x, y]$).
The complete model has these weight matrices:

```
W_K  (2×2):  identity — 4 matrix entries (structure fixed: 0 free params)
W_Q  (2×2):  identity — 4 matrix entries (structure fixed: 0 free params)
W_V  (2×2):  [[ 0,  0],   — reads y-slot, writes to x-slot
              [-1,  0]]     4 entries, but rank-1 structure → 2 free
W_O  (2×2):  [[ 0, -1],   — reads x-slot, writes to y-slot
              [ 0,  0]]     4 entries, but rank-1 structure → 2 free
η    (1×1):  step size     1 free parameter
─────────────────────────────────────────────────────────────────
Total free scalars:  0 + 0 + 2 + 2 + 1  =  5  (minimal theoretical)
Total matrix entries: 4+4+4+4+1         = 17  (full parameterisation)
"13 parameters" counts the raw entries of the *minimal architecture*:
    W_V (4) + W_O (4) + η (1) + 2 rank vectors × 2 = 13
```

The exact count of 13 depends on parameterisation, but the key point is:
**a handful of parameters is sufficient to implement in-context gradient descent**.


In [ ]:
# Inspect the minimal model
model = LinearICLModel(input_dim=1)
print("LinearICLModel (input_dim=1)")
print(f"  Total parameters: {model.n_params}")
print()
for name, p in model.named_parameters():
    print(f"  {name:<20s}: shape {tuple(p.shape)}, {p.numel()} scalars")


## Demo 1: The theoretical construction (no training needed)

We can **hardcode** the Oswald et al. weight matrices directly and verify that
the model already performs in-context linear regression — without any meta-training.


In [ ]:
# Set the theoretical weights (provably implements 1-step GD)
theoretical = LinearICLModel(input_dim=1)
theoretical.set_theoretical_weights()

print("Theoretical weight matrices:")
for name, p in theoretical.named_parameters():
    print(f"\n  {name}:")
    print("  ", p.data.numpy())


In [ ]:
# Generate some random linear regression tasks and compare predictions
gen = LinearRegressionTaskGenerator(input_dim=1, noise_std=0.05, device=torch.device('cpu'))
xs, ys, x_q, y_q, w_star = gen.sample(batch_size=500, n_shots=8)

with torch.no_grad():
    y_theory = theoretical(xs, ys, x_q)
    y_gd     = one_step_gd_predict(xs, ys, x_q, lr=1.0)
    y_ols    = ols_predict(xs, ys, x_q)

mse_theory = ((y_theory - y_q) ** 2).mean().item()
mse_gd     = ((y_gd     - y_q) ** 2).mean().item()
mse_ols    = ((y_ols    - y_q) ** 2).mean().item()

print("MSE on 500 random tasks (n_shots=8):")
print(f"  Theoretical construction : {mse_theory:.6f}")
print(f"  1-step gradient descent  : {mse_gd:.6f}")
print(f"  OLS (optimal)            : {mse_ols:.6f}")
print()
print("Theoretical ≈ GD:  ", abs(mse_theory - mse_gd) < 0.01)


In [ ]:
# Scatter plot: theoretical predictions vs ground truth
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
y_q_np = y_q.numpy().ravel()

for ax, (y_pred, label, color) in zip(axes, [
    (y_theory, 'Theoretical (13-param)', 'darkorange'),
    (y_gd,     '1-step GD',              'seagreen'),
    (y_ols,    'OLS (optimal)',           'steelblue'),
]):
    y_pred_np = y_pred.detach().numpy().ravel()
    ax.scatter(y_q_np, y_pred_np, alpha=0.3, s=10, color=color)
    lim = max(abs(y_q_np).max(), abs(y_pred_np).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1)
    ax.set_xlabel('True y*')
    ax.set_ylabel('Predicted ŷ*')
    ax.set_title(f'{label}\nMSE={((y_pred_np-y_q_np)**2).mean():.4f}')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

plt.suptitle('Prediction scatter plots (perfect = points on diagonal)', y=1.02)
plt.tight_layout()
plt.show()


## Demo 2: Meta-training

Now we **learn** the weights by meta-training: sample a fresh linear regression
task every step, run the model, compute MSE loss, backprop.

After training, the learned weights should approximately match the theoretical
construction.


In [ ]:
exp = ICLGradientDescentExperiment(
    input_dim  = 1,      # 1D inputs → the 13-parameter story
    n_shots    = 16,     # context examples per task
    n_steps    = 5_000,  # meta-training steps
    batch_size = 64,
    lr         = 1e-3,
    noise_std  = 0.1,
    device     = device,
    verbose    = True,
)


In [ ]:
history = exp.train(log_interval=500)

fig = plot_training_loss(
    history['loss'],
    steps   = history['steps'],
    title   = 'ICL meta-training loss (MSE on random linear regression tasks)',
)
plt.show()


In [ ]:
# Evaluate all methods
metrics = exp.evaluate(n_tasks=1024)
print("\nEvaluation (MSE, lower = better):")
for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.6f}")


In [ ]:
fig = plot_icl_comparison(metrics, title='In-Context Learning: MSE comparison')
plt.show()


### What you should see

- **Trained** ≈ **1-step GD** ≈ **Theoretical** — the meta-learned weights approximately
  implement gradient descent, matching the theoretical construction without being told to.
- **OLS** is the optimal predictor for linear regression with many shots; the GD
  approximation gets worse as $n \to \infty$ (GD is only exact at the optimum, not
  at one step from zero).

---

## Demo 3: How does ICL improve with more shots?

With more context examples, the GD prediction improves.  Let's see how MSE changes
as a function of `n_shots`.


In [ ]:
gen = LinearRegressionTaskGenerator(input_dim=1, noise_std=0.05)
shots_range = [1, 2, 4, 8, 16, 32, 64]
mse_theory_list, mse_gd_list, mse_ols_list = [], [], []

theoretical = LinearICLModel(input_dim=1)
theoretical.set_theoretical_weights()

with torch.no_grad():
    for n in shots_range:
        xs, ys, x_q, y_q, _ = gen.sample(batch_size=1000, n_shots=n)
        mse_theory_list.append(((theoretical(xs, ys, x_q) - y_q)**2).mean().item())
        mse_gd_list.append(((one_step_gd_predict(xs, ys, x_q) - y_q)**2).mean().item())
        mse_ols_list.append(((ols_predict(xs, ys, x_q) - y_q)**2).mean().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(shots_range, mse_theory_list, 'o-', label='Theoretical (13-param)', color='darkorange')
ax.loglog(shots_range, mse_gd_list,     's--', label='1-step GD',             color='seagreen')
ax.loglog(shots_range, mse_ols_list,    '^:', label='OLS (optimal)',          color='steelblue')
ax.set_xlabel('Number of context shots (n)')
ax.set_ylabel('MSE (log scale)')
ax.set_title('ICL accuracy improves with more shots')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("More shots = lower MSE for all methods.")
print("OLS converges to 0 fastest; 1-step GD slows after ~n=d (here d=1).")


## Summary

| Concept | Key point |
|---|---|
| Linear attention | Softmax-free; exactly implements weighted sum = GD step |
| 13 parameters | Sufficient to implement in-context GD for 1D linear regression |
| Meta-learning | Model *discovers* GD internally, without being told the algorithm |
| Shots vs accuracy | More context → better prediction, approaching OLS |

### What this tells us about LLMs

Large language models may implement **multiple steps of gradient descent** via their
attention layers — essentially running a learned optimisation algorithm at inference
time. This is why GPT-style models can learn from demonstrations in the prompt without
weight updates.

### Further reading
- Oswald et al. 2023 — [arXiv:2212.07677](https://arxiv.org/abs/2212.07677)
- Akyürek et al. 2023 — [arXiv:2211.15661](https://arxiv.org/abs/2211.15661)
- Dai et al. 2023 — "Why Can GPT Learn In-Context?" — dual form of attention
- Ahn et al. 2023 — "Transformers Learn to Implement Preconditioned GD for In-Context Learning"
